In [ ]:
import pandas as pd
import os

In [ ]:
csv_folder = "data/elementor_data"
csv_files = [os.path.join(csv_folder, f) for f in os.listdir(csv_folder) if f.endswith('.csv')]
dfs = []
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        dfs.append(df)
    except Exception as e:
        print(f"Erro ao ler {csv_file}: {e}")

if dfs:
    merged_df = pd.concat(dfs, ignore_index=True)
else:
    merged_df = pd.DataFrame()


In [ ]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(merged_df):
    # Change column type to datetime64[ns] for column: 'Created At'
    merged_df = merged_df.astype({'Created At': 'datetime64[ns]'})
    # Rename column 'Unnamed: 0' to 'name'
    merged_df = merged_df.rename(columns={'Unnamed: 0': 'name'})
    # Rename column 'Unnamed: 1' to 'email'
    merged_df = merged_df.rename(columns={'Unnamed: 1': 'email'})
    # Rename column 'Unnamed: 2' to 'phone'
    merged_df = merged_df.rename(columns={'Unnamed: 2': 'phone'})
    # Sort by column: 'Created At' (descending)
    merged_df = merged_df.sort_values(['Created At'], ascending=[False])
    # Drop columns: 'conversion_type_id', 'conversion_date' and 4 other columns
    merged_df = merged_df.drop(columns=['conversion_type_id', 'conversion_date', 'Form Name (ID)', 'User ID', 'User Agent', 'User IP'])
    return merged_df

merged_df_clean = clean_data(merged_df.copy())
merged_df_clean.head()

In [ ]:
from urllib.parse import urlparse, parse_qs

# Função para preencher utms faltantes a partir da query string de 'Referrer'
def fill_utm_from_referrer(row):
    # Só tentar preencher se utm_source está faltando OU NA (None, NaN ou string vazia)
    cond = pd.isna(row['utm_source']) or row['utm_source'] in ['', None]
    if cond:
        ref_url = row.get('Referrer', '')
        if isinstance(ref_url, str) and ref_url.startswith('http'):
            try:
                parsed_url = urlparse(ref_url)
                query_params = parse_qs(parsed_url.query)
                # Para cada utm, tenta extrair da query e popular se vazio ou NA
                for utm_key in ['utm_source','utm_medium','utm_content','utm_campaign','utm_term']:
                    value = query_params.get(utm_key, [None])[0]
                    if pd.isna(row.get(utm_key)) or row.get(utm_key) in ['', None]:
                        row[utm_key] = value
            except Exception as e:
                pass  # Ignora caso dê problema em URL
    return row

# Aplica a função linha-a-linha
merged_df_clean = merged_df_clean.apply(fill_utm_from_referrer, axis=1)


In [ ]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(merged_df_clean):
    # Filter rows based on column: 'utm_source'
    merged_df_clean = merged_df_clean[(merged_df_clean['utm_source'].str.contains("meta-ads", regex=False, na=False, case=False)) | (merged_df_clean['utm_source'].str.contains("fb", regex=False, na=False, case=False))]
    return merged_df_clean

merged_df_clean_1 = clean_data(merged_df_clean.copy())
merged_df_clean_1.head()